# LoRA fine-tune a small LLM on a T4

Qwen2.5-0.5B-Instruct + LoRA on 1,000 Alpaca examples. About 5 minutes on a free T4.

Kernel: **Select Kernel → Colab → New Colab Server → GPU → T4**. A CPU server will stop at the first cell.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: reconnect with a T4 Colab server"
print(torch.cuda.get_device_name(0))

In [ ]:
# torch, transformers, datasets and accelerate come preinstalled on Colab.
# Colab also ships torchao 0.10, which the latest peft rejects; this notebook doesn't use torchao, so remove it.
!pip -q install peft
!pip -q uninstall -y torchao

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
# T4 has no fast bf16, so keep weights fp32 and let the Trainer use fp16 mixed precision
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to("cuda")

def ask(q, max_new_tokens=128):
    text = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to("cuda")
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

PROBE = "Give three tips for staying healthy."
print("BEFORE:\n", ask(PROBE))

In [ ]:
from datasets import load_dataset

ds = load_dataset("yahma/alpaca-cleaned", split="train[:1000]")

def to_tokens(ex):
    user = ex["instruction"] + ("\n\n" + ex["input"] if ex["input"] else "")
    msgs = [{"role": "user", "content": user}, {"role": "assistant", "content": ex["output"]}]
    return tok(tok.apply_chat_template(msgs, tokenize=False), truncation=True, max_length=512)

ds = ds.map(to_tokens, remove_columns=ds.column_names)
ds

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

# ponytail: loss covers prompt + answer; mask prompt tokens if answer quality matters
trainer = Trainer(
    model=model,
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    args=TrainingArguments(
        output_dir="/content/out",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
model.eval()
print("AFTER:\n", ask(PROBE))

In [ ]:
# The Colab runtime is wiped when it disconnects, so zip the adapter (a few MB) and download it
model.save_pretrained("/content/qwen-lora")
tok.save_pretrained("/content/qwen-lora")
!cd /content && zip -qr qwen-lora.zip qwen-lora && ls -lh qwen-lora.zip

In [ ]:
# Download link for VS Code: files.download() doesn't work there, and widgets render empty without CDN access.
# The zip is embedded in the link as base64, so clear this cell's output before saving the notebook.
# ponytail: whole file lives in the output, fine for MB-sized adapters, not for full models.
import base64
from IPython.display import HTML

b64 = base64.b64encode(open("/content/qwen-lora.zip", "rb").read()).decode()
HTML(f'<a download="qwen-lora.zip" href="data:application/zip;base64,{b64}">Download qwen-lora.zip</a>')

In [ ]:
# Upload button for VS Code: files.upload() doesn't work there; the colab-vscode wiki suggests the FileUpload widget.
# Pick 01-unknown-code.png and it is saved to /content as soon as it arrives.
import ipywidgets as w

up = w.FileUpload(accept=".png", multiple=False)

def save(change):
    v = change["new"]
    # ipywidgets 7 (Colab today) gives {name: {...}}; ipywidgets 8 gives a tuple of dicts with "name"
    items = v.items() if isinstance(v, dict) else ((f["name"], f) for f in v)
    for name, f in items:
        with open(f"/content/{name}", "wb") as out:
            out.write(bytes(f["content"]))
        print("saved", f"/content/{name}")

up.observe(save, names="value")
up